# gemma4-baseline-v1 — adapter-free diagnostic on `gemma-4-31b-it-qat-w4a16-ct`

The first run of this notebook used a LoRA adapter and scored **0 of 129**. This revision runs the
artifact that would actually be graded: the declarative submission, **no adapter**.

Three defects made the old run worthless, and each one is now an assertion rather than a comment:

| Old defect | Consequence | Now enforced by |
|---|---|---|
| `thinking_level` in `sampling.yaml` | LiteLLM sent `reasoning_effort: medium`, Gemma bridge raised `RuntimeError` → 115/129 tasks crashed on turn 1 with 0 tokens | Phase 4.5 `sampling_yaml_has_no_thinking_level` |
| LoRA trained on `unsloth/…bnb-4bit`, served on `…qat-w4a16-ct` | unvalidated artifact in the graded path | Phase 4.5 `agent_yaml_has_no_adapter_key` + `submission_ships_no_adapter_files` |
| run measured an inline reconstruction, not the artifact | the number described something unshipped | Phase 3 unpacks `submission.zip`; Phase 4.5 asserts its sha256 |

**The run is capped at 20 stratified tasks.** One task is spent first on a non-degeneracy smoke
test (Phase 4.6). If the model returns zero completion tokens on that one task, the campaign
aborts with a `DEGENERATE` marker instead of spending ~1.7 h to rediscover the 0/129 signature.

**Phase timing is recorded only when the harness actually reports it.** Columns that the harness
does not expose are `null`, never inferred from wall-clock arithmetic — see Phase 5.

**Budget comes from the shipped `eval_config.yaml`**, not from constants in this notebook.

In [ ]:
# ===========================================================================
# RUN POLICY FLAG -- read this before launching the diagnostic run.
# ---------------------------------------------------------------------------
# TASK_LIMIT caps the diagnostic at a stratified sample of the 129-task corpus.
# The point of the FIRST pass is to answer one question cheaply: does
# gemma-4-31b-it-qat-w4a16-ct generate anything at all? Every run on record
# resolved 0 tasks, so a 20-task sample that returns non-zero
# total_completion_tokens is worth 100x the GPU time of a 129-task run that
# returns zero again. Expand to 129 ONLY after this pass is non-degenerate.
# Set to 0 to run the full corpus.
# ---------------------------------------------------------------------------
# STRATIFIED_MIX is the sample shape. It mirrors the corpus mix in
# docs/02-dataset.md:13 so the sample is not dominated by one library.
# Tasks are taken in corpus order within each repo, so the selection is
# deterministic and reproducible -- not a random draw that changes per run.
# ---------------------------------------------------------------------------
# ALLOW_PARTIAL_PREFLIGHT decides what happens when a HARD infrastructure
# precondition fails (see Phase 4.5 / 4.6).
#
#   False (DEFAULT) -> ABORT. Preflight writes preflight_report.json, prints a
#                      loud banner, and raises SystemExit(1) BEFORE the task
#                      loop starts. Zero GPU hours are spent on a run whose
#                      number would be meaningless.
#
#   True            -> CONTINUE past the failures anyway, but every failure is
#                      recorded in preflight_report.json and a prominent warning
#                      banner prints at the START and again at the END of the
#                      run. This is a deliberate operator override; it is never
#                      the default.
# ---------------------------------------------------------------------------
# Do NOT flip this to True to silence a failing banner. Flip it only when you
# have consciously decided that spending scarce GPU quota on a knowingly
# degraded run is worth the diagnostic.
# ===========================================================================
ALLOW_PARTIAL_PREFLIGHT = False

# 20 tasks: 8 fastapi, 7 rich, 4 requests, 1 httpx.
TASK_LIMIT = 20
STRATIFIED_MIX = [("fastapi", 8), ("rich", 7), ("requests", 4), ("httpx", 1)]

In [ ]:
# ===========================================================================
# PHASE 0 -- Banner, input discovery, and the shared run context.
# Prints a timestamp and the RESOLVED input paths so the log alone proves
# which datasets actually mounted. An operator reading only this log must be
# able to tell exactly what was measured.
#
# NOTE ON THE ADAPTER: this kernel deliberately mounts NO LoRA. The adapter was
# purged from the submission on 2026-10-05 (unvalidated: training loss 3.51 vs a
# documented < 1.1 target, eval_steps 500 > max_steps 60 so validation never
# ran, and 16 training rows whose observation IS a SYNTAX_ERROR). What ships is
# the declarative tree only. See docs/FINDINGS.md.
# ===========================================================================
import datetime
import glob
import hashlib
import importlib
import inspect
import json
import os
import platform
import shutil
import subprocess
import sys
import time
import traceback
from pathlib import Path

NOTEBOOK_NAME = "baseline_v1.ipynb"
KERNEL_SLUG = "francisclyap/gemma4-baseline-v1"
MODEL_ID = "gemma-4-31b-it-qat-w4a16-ct"

# sha256 of the submission.zip this diagnostic is supposed to measure. If the
# artifact is rebuilt, this constant must be updated with it -- otherwise the
# run measures something other than what would be graded.
SUBMISSION_ZIP_SHA256 = "bd4f31cd7971ef4df6910f6c5324259d8d337cc527b2f67549340a2f0745b05e"

RUN_TS = datetime.datetime.now(datetime.timezone.utc).isoformat()

INPUT_ROOT = Path("/kaggle/input")
WORKING_DIR = Path("/kaggle/working")
WORKING_DIR.mkdir(parents=True, exist_ok=True)

print("=" * 78)
print("  gemma4-baseline-v1 : adapter-free diagnostic run")
print("=" * 78)
print(f"  notebook      : {NOTEBOOK_NAME}")
print(f"  kernel slug   : {KERNEL_SLUG}")
print(f"  started (UTC) : {RUN_TS}")
print(f"  python        : {sys.version.split()[0]} ({platform.platform()})")
print(f"  input root    : {INPUT_ROOT}")
print(f"  working dir   : {WORKING_DIR}")
print(f"  model         : {MODEL_ID}")
print(f"  adapter       : NONE (shipped submission declares no adapter)")
print("=" * 78)

# ---- prove which inputs mounted -------------------------------------------
print("\n--- /kaggle/input (resolved) ---")
if INPUT_ROOT.is_dir():
    for child in sorted(INPUT_ROOT.iterdir()):
        print(f"  {child}")
else:
    print("  /kaggle/input is absent; this notebook expects the Kaggle runtime.")
print("=" * 78, flush=True)

## Phase 1 — Offline install from the wheelhouse dataset

The kernel has `enable_internet: false`, so **there is no network at runtime**.
Every Python dependency must come from a wheel file already sitting in an
attached dataset.

We do **not** hardcode a directory name such as `.../wheelhouse`. Kaggle's mount
layout for a dataset is `<owner>/<dataset-slug>`, and slug normalisation is not
guaranteed to match what we typed in `kernel-metadata.json`. Instead we **glob
`/kaggle/input` for a directory that actually contains `.whl` files**.

`pip install --no-index --find-links=<that dir>` then installs strictly from
local files: no index lookup, no network, no surprise.

In [ ]:
# ===========================================================================
# PHASE 1 -- Discover the wheelhouse by globbing, then install offline.
# ===========================================================================
def discover_wheelhouse():
    """Return every /kaggle/input directory that actually holds .whl files.

    Globbing beats hardcoding: the mount path Kaggle creates is derived from the
    dataset slug and is not guaranteed to match the string we wrote in
    kernel-metadata.json.
    """
    hits = []
    if not INPUT_ROOT.is_dir():
        return hits
    for whl in INPUT_ROOT.rglob("*.whl"):
        parent = whl.parent
        if parent not in hits:
            hits.append(parent)
    # Prefer a path that looks like the wheelhouse dataset.
    hits.sort(key=lambda p: (0 if "wheelhouse" in str(p).lower() else 1, str(p)))
    return hits


wheelhouse_dirs = discover_wheelhouse()
print("wheelhouse directories discovered by glob:")
for path in wheelhouse_dirs:
    count = len(list(path.glob("*.whl")))
    print(f"  {path}  ({count} wheels)")
if not wheelhouse_dirs:
    print("  WARNING: no .whl files found under /kaggle/input.")
    print("  The harness cannot be imported. Later phases will record failures.")

PRIMARY_WHEELHOUSE = wheelhouse_dirs[0] if wheelhouse_dirs else None

if PRIMARY_WHEELHOUSE is not None:
    wheels = sorted(str(p) for p in PRIMARY_WHEELHOUSE.glob("*.whl"))
    print(f"\ninstalling {len(wheels)} wheels from {PRIMARY_WHEELHOUSE} (offline)")
    cmd = [
        sys.executable, "-m", "pip", "install",
        "--no-index",
        f"--find-links={PRIMARY_WHEELHOUSE}",
        "--upgrade",
        *wheels,
    ]
    print("  $ " + " ".join(cmd[:6]) + f" ... ({len(wheels)} wheels)")
    proc = subprocess.run(cmd, capture_output=True, text=True)
    print(f"  pip exit code: {proc.returncode}")
    if proc.stdout.strip():
        tail = proc.stdout.strip().splitlines()[-12:]
        for line in tail:
            print("    " + line)
    if proc.returncode != 0:
        print("  pip stderr tail:")
        for line in proc.stderr.strip().splitlines()[-12:]:
            print("    " + line)

importlib.invalidate_caches()
print("\nPhase 1 complete.", flush=True)

## Phase 2 — Import the harness and print its real API surface

There is no PyPI package named `swegemma`. The harness module inside Kaggle is
called **`adk_submission`** and arrives from the wheelhouse dataset.

We import it, report its version, and **enumerate the task-loading and
evaluation entry points it actually exposes** via `dir()` and, where possible,
`inspect.signature`. Printing this is not decoration: it is how the log reveals
the true names, so that a later failure can be attributed to a renamed API
rather than to guesswork in this notebook.

**If the import fails we print the FULL traceback and re-raise.** A missing
harness is a hard stop. There is nothing to measure without it, and silently
carrying on would produce an empty run that reads like a legitimate zero.

In [ ]:
# ===========================================================================
# PHASE 2 -- Import adk_submission; report version and real entry points.
# ===========================================================================
API_SURFACE_PATH = WORKING_DIR / "api_surface.txt"
api_surface_lines = []


def record(line=""):
    api_surface_lines.append(line)
    print(line, flush=True)


# Preflight (Phase 4.5) needs to know whether the harness actually imported,
# rather than re-deriving it. Both are declared before the try so the names
# always exist for the report, whatever happens next.
ADK_SUBMISSION_IMPORTED = False
ADK_SUBMISSION_FILE = None

try:
    import adk_submission

    ADK_SUBMISSION_IMPORTED = True
    ADK_SUBMISSION_FILE = getattr(adk_submission, "__file__", None)

    record("=" * 78)
    record("  adk_submission API SURFACE (discovered at runtime, not assumed)")
    record("=" * 78)
    record(f"  module file : {getattr(adk_submission, '__file__', '<unknown>')}")
    version = getattr(adk_submission, "__version__", None)
    record(f"  __version__ : {version}")
    if version is None:
        record("  (module exposes no __version__ -- reporting 'unknown')")

    record("\n  dir(adk_submission):")
    for name in sorted(dir(adk_submission)):
        if not name.startswith("__"):
            record(f"    {name}")

    dunder_all = getattr(adk_submission, "__all__", None)
    record(f"\n  __all__ : {dunder_all}")

    # Signatures for the things this notebook actually intends to call.
    record("\n  signatures for task-loading / evaluation entry points:")
    for name in sorted(dir(adk_submission)):
        if name.startswith("_"):
            continue
        obj = getattr(adk_submission, name, None)
        if not (inspect.isfunction(obj) or inspect.isclass(obj)):
            continue
        try:
            record(f"    {name}{inspect.signature(obj)}")
        except (TypeError, ValueError):
            record(f"    {name}  <signature unavailable>")

    record("=" * 78)
except Exception:
    record("=" * 78)
    record("  adk_submission IMPORT FAILED -- full traceback follows")
    record("=" * 78)
    for line in traceback.format_exc().rstrip().splitlines():
        record("  " + line)
    record("=" * 78)
    # Re-raise: a kernel with no harness measures nothing. Continuing would
    # emit an empty result set indistinguishable from a real zero score.
    raise

API_SURFACE_PATH.write_text("\n".join(api_surface_lines) + "\n")
print(f"\nAPI surface written to {API_SURFACE_PATH}", flush=True)

## Phase 3 — Unpack and verify the shipped submission artifact

The point of this run is to measure **the artifact that would actually be graded**, so the
diagnostic loads `submission.zip` itself rather than rebuilding an equivalent agent inline. A
measurement of a hand-built approximation is evidence about the approximation.

Three things are asserted here and re-asserted in Phase 4.5:

- the archive's sha256 matches the one this kernel was built against;
- `agent.yaml` declares **no** `adapter:` key;
- `configs/sampling.yaml` contains **no** `thinking_level` key — the exact key that made LiteLLM
  emit `reasoning_effort: medium` and crash the Gemma bridge on turn 1 of 115/129 tasks.

In [ ]:
# ===========================================================================
# PHASE 3 -- Locate, verify and unpack the shipped submission.zip.
# ===========================================================================
SUBMISSION_DIR = WORKING_DIR / "submission"


def sha256_of(path, chunk=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as handle:
        for block in iter(lambda: handle.read(chunk), b""):
            digest.update(block)
    return digest.hexdigest()


def locate_submission_zip():
    """Find submission.zip wherever the runtime happens to place it.

    Kaggle derives its mount paths from dataset slugs, so the location is not
    something we get to choose. Kernel source files and dataset mounts are both
    searched; the first hit wins. If none is found the preflight FAILS loudly,
    which is the correct outcome -- silently measuring an inline agent is how a
    run ends up describing something that is not shipped.
    """
    for root in (WORKING_DIR, INPUT_ROOT, Path("/kaggle")):
        if not root.is_dir():
            continue
        for hit in sorted(root.rglob("submission.zip")):
            return hit
    return None


SUBMISSION_ZIP_PATH = locate_submission_zip()
OBSERVED_SUBMISSION_SHA256 = None
SHIPPED_FILES = []
SHIPPED_AGENT_YAML = {}
SHIPPED_SAMPLING_YAML = {}
SHIPPED_EVAL_CONFIG_YAML = {}
SHIPPED_PROMPT = ""

print("=" * 78)
if SUBMISSION_ZIP_PATH is None:
    print("  SUBMISSION.ZIP NOT FOUND")
    print("=" * 78)
    print("  Expected submission.zip in /kaggle/working (kernel source) or under")
    print("  /kaggle/input. Without it this run cannot measure the shipped")
    print("  artifact. Preflight will FAIL and the run will abort.")
    print("=" * 78, flush=True)
else:
    import zipfile

    print("  submission.zip :", SUBMISSION_ZIP_PATH)
    OBSERVED_SUBMISSION_SHA256 = sha256_of(SUBMISSION_ZIP_PATH)
    print("  sha256         :", OBSERVED_SUBMISSION_SHA256)
    print("  expected       :", SUBMISSION_ZIP_SHA256)
    if OBSERVED_SUBMISSION_SHA256 != SUBMISSION_ZIP_SHA256:
        print("=" * 78)
        print("  ##################  LOUD WARNING  ##################")
        print("  SUBMISSION SHA256 MISMATCH")
        print("    expected :", SUBMISSION_ZIP_SHA256)
        print("    observed :", OBSERVED_SUBMISSION_SHA256)
        print("  This run is NOT measuring the artifact named in the constant")
        print("  above. Scores will not describe the shipped submission.")
        print("=" * 78)
    else:
        print("  SHA256: MATCH -- this is the intended artifact.")

    if SUBMISSION_DIR.exists():
        shutil.rmtree(SUBMISSION_DIR)
    SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(SUBMISSION_ZIP_PATH) as archive:
        archive.extractall(SUBMISSION_DIR)
        SHIPPED_FILES = sorted(
            n for n in archive.namelist() if not n.endswith("/")
        )
    print(f"\n  unpacked {len(SHIPPED_FILES)} files into {SUBMISSION_DIR}")
    for name in SHIPPED_FILES:
        size = (SUBMISSION_DIR / name).stat().st_size
        print(f"      {name} ({size:,} bytes)")

    adapter_members = [n for n in SHIPPED_FILES if n.startswith("adapters/")]
    print("\n  adapter members in the archive :", len(adapter_members) or "none")

    # The submission is YAML with a custom !include tag; a safe loader must be
    # supplied or PyYAML raises ConstructorError on `instruction: !include ...`.
    class _IncludeLoader(__import__("yaml").SafeLoader):
        pass

    def _include_constructor(loader, node):
        return {"__include__": loader.construct_scalar(node)}

    _IncludeLoader.add_constructor("!include", _include_constructor)

    def _read_yaml(path):
        try:
            return __import__("yaml").load(path.read_text(), Loader=_IncludeLoader)
        except Exception as exc:
            print(f"  WARNING: could not parse {path.name}: {exc}")
            return {}

    agent_yaml_path = SUBMISSION_DIR / "agent.yaml"
    sampling_yaml_path = SUBMISSION_DIR / "configs" / "sampling.yaml"
    eval_cfg_yaml_path = SUBMISSION_DIR / "eval_config.yaml"
    prompt_path = SUBMISSION_DIR / "prompts" / "main.md"

    SHIPPED_AGENT_YAML = _read_yaml(agent_yaml_path) if agent_yaml_path.is_file() else {}
    SHIPPED_SAMPLING_YAML = _read_yaml(sampling_yaml_path) if sampling_yaml_path.is_file() else {}
    SHIPPED_EVAL_CONFIG_YAML = _read_yaml(eval_cfg_yaml_path) if eval_cfg_yaml_path.is_file() else {}
    SHIPPED_PROMPT = prompt_path.read_text() if prompt_path.is_file() else ""

    print("\n  --- shipped config ---")
    print("  agent.yaml model   :", SHIPPED_AGENT_YAML.get("model"))
    print("  agent.yaml adapter :", SHIPPED_AGENT_YAML.get("adapter", "<absent>"))
    print("  sampling.yaml      :", json.dumps(SHIPPED_SAMPLING_YAML, sort_keys=True))
    print("  eval_config.yaml   :", json.dumps(SHIPPED_EVAL_CONFIG_YAML, sort_keys=True))
    print("  prompt chars       :", len(SHIPPED_PROMPT))
print("=" * 78, flush=True)

## Phase 4 — Load the corpus, then take a stratified sample

The full 129-task corpus is loaded first (Phase 4.5 asserts it is exactly 129 — that is the check
that the competition data actually mounted). Only then is the diagnostic sample drawn: **8 fastapi,
7 rich, 4 requests, 1 httpx**, matching the corpus mix in `docs/02-dataset.md:13`.

Selection is the *first* N tasks of each repo **in corpus order**, not a random draw. A random
sample would differ between runs and make two runs' numbers incomparable, which defeats the
purpose of having a baseline to compare against.

In [ ]:
# ===========================================================================
# PHASE 4 -- Load the corpus, then take a deterministic stratified sample.
# ===========================================================================
DATA_DIR_CANDIDATES = [
    Path("/kaggle/input/competitions/gemma-4-developer-agent"),
]
if INPUT_ROOT.is_dir():
    DATA_DIR_CANDIDATES.extend(
        sorted(p for p in INPUT_ROOT.glob("competitions/*") if p.is_dir())
    )

DATA_DIR = next((p for p in DATA_DIR_CANDIDATES if (p / "tasks.jsonl").is_file()), DATA_DIR_CANDIDATES[0])
TASKS_PATH = DATA_DIR / "tasks.jsonl"

print("competition data dir :", DATA_DIR)
print("tasks.jsonl          :", TASKS_PATH, "| exists:", TASKS_PATH.is_file())

all_tasks = []
load_method = "unresolved"

try:
    from swegemma.models import load_tasks

    all_tasks = list(load_tasks(TASKS_PATH))
    load_method = "swegemma.models.load_tasks"
except Exception:
    print("\n  swegemma.models.load_tasks unavailable; full traceback:")
    for line in traceback.format_exc().rstrip().splitlines():
        print("    " + line)
    try:
        with open(TASKS_PATH) as handle:
            all_tasks = [json.loads(line) for line in handle if line.strip()]
        load_method = "json-lines fallback"
    except Exception:
        for line in traceback.format_exc().rstrip().splitlines():
            print("    " + line)


def task_field(task, name, default=None):
    if isinstance(task, dict):
        return task.get(name, default)
    return getattr(task, name, default)


CORPUS_SIZE = len(all_tasks)
print(f"\nCORPUS: {CORPUS_SIZE} tasks (loaded via {load_method})")

repo_histogram = {}
for task in all_tasks:
    key = str(task_field(task, "repo"))
    repo_histogram[key] = repo_histogram.get(key, 0) + 1
print("corpus mix:", json.dumps(dict(sorted(repo_histogram.items(), key=lambda kv: -kv[1])), indent=None))


def select_stratified(corpus, mix):
    """Take the first N tasks of each repo, in corpus order.

    Deterministic on purpose: a random draw would change every run and make two
    runs' numbers incomparable. Repos named in `mix` but absent from the corpus
    are reported rather than silently ignored.
    """
    buckets = {}
    for task in corpus:
        buckets.setdefault(str(task_field(task, "repo")), []).append(task)

    picked = []
    missing = []
    for repo, count in mix:
        available = buckets.get(repo, [])
        if not available:
            missing.append(repo)
            continue
        if len(available) < count:
            print(f"  WARNING: repo {repo!r} has only {len(available)} tasks, wanted {count}")
        picked.extend(available[:count])

    # Top up from whatever is left if a requested repo was short, so the sample
    # size the operator asked for is the size actually run.
    if TASK_LIMIT and len(picked) < TASK_LIMIT:
        seen = {id(t) for t in picked}
        for task in corpus:
            if len(picked) >= TASK_LIMIT:
                break
            if id(task) not in seen:
                picked.append(task)
                seen.add(id(task))

    if missing:
        print(f"  WARNING: repos absent from corpus, requested in mix: {missing}")
    return picked


if TASK_LIMIT:
    tasks = select_stratified(all_tasks, STRATIFIED_MIX)[:TASK_LIMIT]
else:
    tasks = list(all_tasks)

selected_histogram = {}
for task in tasks:
    key = str(task_field(task, "repo"))
    selected_histogram[key] = selected_histogram.get(key, 0) + 1

print(f"\nSELECTED: {len(tasks)} of {CORPUS_SIZE} tasks")
print("selected mix:", json.dumps(dict(sorted(selected_histogram.items(), key=lambda kv: -kv[1]))))
print("selected task ids:")
for task in tasks:
    tid = task_field(task, "instance_id") or task_field(task, "task_id")
    print("   ", tid, "|", task_field(task, "repo"))
print(flush=True)

## Phase 4.5 — HARD PREFLIGHT (abort before spending a single GPU hour)

Preconditions only. Default policy is **ABORT**: a failed precondition means every task result
would be meaningless, so we stop before spending GPU hours rather than after producing a number
that gets misread.

Check 3 asserts the **full corpus** loaded (129). That is the check that the competition data
mounted — it is deliberately independent of `TASK_LIMIT`, which only controls the sample.

In [ ]:
# ===========================================================================
# PHASE 4.5 -- HARD PREFLIGHT. Assert every infrastructure precondition BEFORE
# the task loop. Default policy is ABORT: a failed precondition means every
# task result would be meaningless, so we stop before spending GPU hours
# rather than after producing a number that gets misread.
# ===========================================================================
PREFLIGHT_CHECKS = []


def preflight_check(name, expected, observed, passed, detail=""):
    """Record one named precondition with its expected value, observed value and verdict."""
    entry = {
        "name": name,
        "expected": str(expected),
        "observed": str(observed),
        "verdict": "PASS" if passed else "FAIL",
        "detail": detail,
    }
    PREFLIGHT_CHECKS.append(entry)
    return entry


# ---- 1) wheelhouse located and non-empty ------------------------------------
wheel_count = len(list(PRIMARY_WHEELHOUSE.glob("*.whl"))) if PRIMARY_WHEELHOUSE else 0
preflight_check(
    "wheelhouse_non_empty",
    "at least 1 .whl under /kaggle/input",
    f"{wheel_count} wheels in {PRIMARY_WHEELHOUSE}",
    wheel_count >= 1,
    "kernel has enable_internet=false, so this is the only install source",
)

# ---- 2) harness import succeeded --------------------------------------------
preflight_check(
    "adk_submission_imported",
    "adk_submission importable",
    f"imported={ADK_SUBMISSION_IMPORTED} file={ADK_SUBMISSION_FILE}",
    bool(ADK_SUBMISSION_IMPORTED),
    "a missing harness measures nothing",
)

# ---- 3) the FULL corpus mounted (independent of TASK_LIMIT) ------------------
preflight_check(
    "corpus_task_count_is_129",
    "129",
    CORPUS_SIZE,
    CORPUS_SIZE == 129,
    f"loaded via {load_method} into {TASKS_PATH}; any other count means the competition "
    f"source did not mount. Deliberately not TASK_LIMIT ({TASK_LIMIT}).",
)

# ---- 4) the shipped artifact mounted and is the right one -------------------
preflight_check(
    "submission_zip_found",
    "submission.zip located",
    SUBMISSION_ZIP_PATH,
    SUBMISSION_ZIP_PATH is not None,
    "this kernel measures the shipped artifact, not an inline reconstruction",
)
preflight_check(
    "submission_zip_sha256_matches",
    SUBMISSION_ZIP_SHA256,
    OBSERVED_SUBMISSION_SHA256,
    OBSERVED_SUBMISSION_SHA256 == SUBMISSION_ZIP_SHA256,
    "a rebuilt artifact would make this number describe something unshipped",
)

# ---- 5) NON-DEGENERACY: agent.yaml declares no adapter ----------------------
adapter_declared = "adapter" in SHIPPED_AGENT_YAML
preflight_check(
    "agent_yaml_has_no_adapter_key",
    "no 'adapter' key in agent.yaml",
    f"adapter={SHIPPED_AGENT_YAML.get('adapter', '<absent>')}",
    not adapter_declared,
    "the adapter was purged 2026-10-05; shipping one again would reintroduce an "
    "unvalidated artifact trained on proxy-model trajectories",
)
adapter_members = [n for n in SHIPPED_FILES if n.startswith("adapters/")]
preflight_check(
    "submission_ships_no_adapter_files",
    "0 members under adapters/",
    f"{len(adapter_members)} members",
    not adapter_members,
    "an undeclared adapter file in the archive is worse than none at all",
)

# ---- 6) NON-DEGENERACY: sampling.yaml has no thinking_level -----------------
# This is the exact key that caused `RuntimeError: unsupported reasoning_effort`
# on turn 1 of 115/129 tasks: LiteLLM maps thinking_level -> reasoning_effort,
# and the Gemma bridge rejects the value. HARNESS_README.md:149 says omit it.
def find_key(obj, key, path="$"):
    """Locate `key` anywhere in a nested structure; return its dotted path or None."""
    if isinstance(obj, dict):
        if key in obj:
            return path
        for k, v in obj.items():
            hit = find_key(v, key, f"{path}.{k}")
            if hit:
                return hit
    elif isinstance(obj, list):
        for i, v in enumerate(obj):
            hit = find_key(v, key, f"{path}[{i}]")
            if hit:
                return hit
    return None


thinking_level_path = find_key(SHIPPED_SAMPLING_YAML, "thinking_level")
preflight_check(
    "sampling_yaml_has_no_thinking_level",
    "no 'thinking_level' key anywhere in sampling.yaml",
    thinking_level_path or "absent",
    thinking_level_path is None,
    "presence of this key is what produced the 0/129 turn-1 crash signature",
)

# ---- 7) a GPU is actually visible ---------------------------------------------
try:
    import torch

    cuda_ok = bool(torch.cuda.is_available())
    gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "no CUDA device visible"
except Exception as exc:
    cuda_ok = False
    gpu_name = f"torch unusable: {type(exc).__name__}: {exc}"
preflight_check(
    "gpu_cuda_visible",
    "torch.cuda.is_available() is True",
    gpu_name,
    cuda_ok,
    "a CPU-only run would waste the entire quota and score near zero",
)

# ---- 8) the sample is the size we asked for ----------------------------------
preflight_check(
    "selected_sample_non_empty",
    f"{TASK_LIMIT} tasks" if TASK_LIMIT else "full corpus",
    len(tasks),
    len(tasks) > 0,
    "an empty sample produces a 0% rate that means nothing",
)

# ---- report every check -------------------------------------------------------
PREFLIGHT_FAILED = [c for c in PREFLIGHT_CHECKS if c["verdict"] == "FAIL"]

print("=" * 78)
print("  PREFLIGHT RESULTS")
print("=" * 78)
for check in PREFLIGHT_CHECKS:
    print(f"  [{check['verdict']}] {check['name']}")
    print(f"         expected : {check['expected']}")
    print(f"         observed : {check['observed']}")
    if check["detail"]:
        print(f"         detail   : {check['detail']}")
print(f"  {len(PREFLIGHT_CHECKS) - len(PREFLIGHT_FAILED)}/{len(PREFLIGHT_CHECKS)} passed")
print("=" * 78)

# preflight_report.json is written on BOTH the abort and the continue path so the
# artifact exists either way. It is the evidence of what actually mounted.
PREFLIGHT_REPORT_PATH = WORKING_DIR / "preflight_report.json"
PREFLIGHT_REPORT_PATH.write_text(
    json.dumps(
        {
            "notebook": NOTEBOOK_NAME,
            "kernel_slug": KERNEL_SLUG,
            "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
            "allow_partial_preflight": ALLOW_PARTIAL_PREFLIGHT,
            "policy": "ABORT" if not ALLOW_PARTIAL_PREFLIGHT else "CONTINUE (operator override)",
            "all_passed": not PREFLIGHT_FAILED,
            "failed_count": len(PREFLIGHT_FAILED),
            "checks": PREFLIGHT_CHECKS,
            "wheelhouse_dir": str(PRIMARY_WHEELHOUSE) if PRIMARY_WHEELHOUSE else None,
            "model_id": MODEL_ID,
            "adapter": None,
            "corpus_size": CORPUS_SIZE,
            "selected_task_count": len(tasks),
        },
        indent=2,
        sort_keys=True,
    )
    + "\n"
)

if PREFLIGHT_FAILED and not ALLOW_PARTIAL_PREFLIGHT:
    # Loud, self-explanatory banner: the reader must understand that the run is
    # being stopped ON PURPOSE because its output would be meaningless.
    print()
    print("#" * 78)
    print("#   A B O R T E D   --   R U N   N O T   E X E C U T E D".center(78))
    print("#" * 78)
    print("#  Preflight FAILED, so the task loop was never started.")
    print("#")
    print("#  Continuing would spend scarce GPU quota to produce a number that")
    print("#  does not describe the shipped artifact. A wrong number is worse")
    print("#  than no number, because it gets read as model quality.")
    print("#")
    print("#  Failed preconditions:")
    for check in PREFLIGHT_FAILED:
        print(f"#    - {check['name']}")
        print(f"#        expected : {check['expected']}")
        print(f"#        observed : {check['observed']}")
    print("#")
    print("#  Common causes:")
    print("#    - submission_zip_*  -> the zip did not upload with the kernel;")
    print("#                           add it to the kernel directory and re-push.")
    print("#    - agent_yaml_has_no_adapter_key / submission_ships_no_adapter_files")
    print("#                         -> an adapter crept back in; see docs/FINDINGS.md.")
    print("#    - sampling_yaml_has_no_thinking_level")
    print("#                         -> configs/sampling.yaml reintroduced the key")
    print("#                            that crashed 115/129 tasks on turn 1.")
    print("#")
    print("#  To deliberately spend the quota anyway, set")
    print("#      ALLOW_PARTIAL_PREFLIGHT = True")
    print("#  in the policy flag cell near the top and re-run. That path records")
    print("#  every failure above and prints a warning banner at the end too.")
    print("#")
    print(f"#  Evidence written to: {PREFLIGHT_REPORT_PATH}")
    print("#" * 78, flush=True)
    raise SystemExit(1)

if PREFLIGHT_FAILED:
    print()
    print("!" * 78)
    print("!   W A R N I N G   --   R U N   C O N T I N U E D   P A S T".center(78))
    print("!   F A I L E D   P R E F L I G H T   C H E C K S".center(78))
    print("!" * 78)
    print("!  ALLOW_PARTIAL_PREFLIGHT is True, so this run is proceeding DESPITE a")
    print("!  broken infrastructure precondition. Its score may be meaningless.")
    print("!  Failed preconditions:")
    for check in PREFLIGHT_FAILED:
        print(f"!    - {check['name']}: observed {check['observed']}")
    print(f"!  Full evidence: {PREFLIGHT_REPORT_PATH}")
    print("!" * 78, flush=True)
else:
    print(f"  All preflight checks PASSED -- proceeding to the {len(tasks)}-task loop.", flush=True)

print(f"\npreflight report: {PREFLIGHT_REPORT_PATH}", flush=True)

## Phase 4.6 — NON-DEGENERACY SMOKE (one task, before committing the quota)

Every Gemma 4 run on record resolved **0 of 129**. The signature is unambiguous: the model
returns **zero completion tokens**, usually because the turn-1 request crashes. A 20-task run
that reproduces that costs ~1.7 h to learn nothing.

So we pay for exactly one task first and assert the two numbers that must be non-zero:

- `total_completion_tokens > 0` — the model actually generated;
- `tool_calls_used >= 1` — the Gemma 4 tool-call grammar actually parsed.

If either is zero, this cell raises `SystemExit(1)` with a `DEGENERATE` marker. That is not a
crash to retry — it is the 0/129 signature, and no wider run is worth the GPU time until it is
explained.

In [ ]:
# ===========================================================================
# PHASE 4.6 -- NON-DEGENERACY SMOKE RUN.
# One task. Two assertions. Abort the whole campaign if either fails.
# ===========================================================================

# Build the evaluator ONCE, here, so Phase 5 reuses the exact same object. The
# budget comes from the SHIPPED eval_config.yaml rather than being hardcoded --
# a diagnostic that overrides the shipped budget is not measuring the shipped
# agent.
EVAL_SECTION = SHIPPED_EVAL_CONFIG_YAML.get("evaluation", {}) or {}
SHIP_TIMEOUT_SECONDS = EVAL_SECTION.get("timeout_seconds", 60)
SHIP_MAX_TIME_MINUTES = EVAL_SECTION.get("max_time_minutes", 4.5)
SHIP_MAX_TOOL_CALLS = EVAL_SECTION.get("max_tool_calls", 40)
SHIP_MAX_TURNS = EVAL_SECTION.get("max_turns", 100)

print("shipped eval_config.yaml evaluation block:")
print(json.dumps(EVAL_SECTION, indent=2, sort_keys=True))
print(f"  -> timeout_seconds   = {SHIP_TIMEOUT_SECONDS}")
print(f"  -> max_time_minutes  = {SHIP_MAX_TIME_MINUTES}")
print(f"  -> max_tool_calls    = {SHIP_MAX_TOOL_CALLS}")
print(f"  -> max_turns         = {SHIP_MAX_TURNS}")

# Any phase-timing field the harness actually exposes. These are recorded when
# present and left None when not -- an invented split is worse than no split.
PHASE_FIELD_CANDIDATES = (
    ("setup_s", "setup_seconds", "container_setup_seconds", "setup_duration"),
    ("agent_s", "agent_seconds", "agent_elapsed_seconds", "agent_duration"),
    ("verify_s", "verify_seconds", "verification_seconds", "verify_duration"),
)


def build_evaluator(results_dir):
    """Construct the harness Evaluator. Returns (evaluator, error_string)."""
    try:
        from swegemma.config import EvalConfig, build_submission_limits
        from swegemma.evaluate import Evaluator

        limits, gen_constraints = build_submission_limits()
        cfg = EvalConfig(
            tasks_path=TASKS_PATH,
            snapshots_dir=DATA_DIR / "snapshots",
            results_dir=results_dir,
            submission_dir=SUBMISSION_DIR,
            models=None,
            sandbox="subprocess",
            timeout_seconds=SHIP_TIMEOUT_SECONDS,
            max_time_minutes=SHIP_MAX_TIME_MINUTES,
            max_tool_calls=SHIP_MAX_TOOL_CALLS,
            max_turns=SHIP_MAX_TURNS,
            limits=limits,
            generation_constraints=gen_constraints,
            adapter_manifest=None,          # shipped artifact declares no adapter
            graph_dir=str(DATA_DIR / "graphs"),
            embeddings_dir=str(DATA_DIR / "embeddings"),
            wheels_dir=str(DATA_DIR / "wheels"),
            verbose=False,
        )
        return Evaluator(cfg), None
    except Exception as exc:
        return None, f"{type(exc).__name__}: {exc}"


def extract_phase_timings(result):
    """Pull real phase timings off the harness result, if it carries them.

    Returns only keys it actually found. A missing phase is recorded as None
    downstream; it is never back-filled with wall-clock arithmetic.
    """
    found = {}
    sources = []
    if isinstance(result, dict):
        sources.append(result)
        for value in result.values():
            if hasattr(value, "__dict__"):
                sources.append(vars(value))
    elif hasattr(result, "__dict__"):
        sources.append(vars(result))
        sources.append(result)

    for canonical, *aliases in PHASE_FIELD_CANDIDATES:
        for source in sources:
            if not isinstance(source, dict):
                continue
            for alias in aliases:
                value = source.get(alias)
                if isinstance(value, (int, float)):
                    found[canonical] = round(float(value), 2)
                    break
            if canonical in found:
                break
        found.setdefault(canonical, None)
    return found


SMOKE_DIR = WORKING_DIR / "smoke"
SMOKE_DIR.mkdir(parents=True, exist_ok=True)

evaluator, evaluator_error = build_evaluator(SMOKE_DIR)
print("\nevaluator:", type(evaluator).__name__ if evaluator else f"UNAVAILABLE ({evaluator_error})")

SMOKE_RESULT = None
if evaluator is not None:
    def _run_smoke():
        return evaluator.evaluate_task(task=tasks[0], task_index=0, total_tasks=1)

    try:
        import asyncio

        try:
            asyncio.get_running_loop()
        except RuntimeError:
            SMOKE_RESULT = asyncio.run(_run_smoke()) if asyncio.iscoroutinefunction(_run_smoke) else _run_smoke()
        else:
            import concurrent.futures

            with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
                SMOKE_RESULT = pool.submit(lambda: asyncio.run(_run_smoke())).result()
    except Exception as exc:
        print("smoke run raised:", type(exc).__name__, exc)
        for line in traceback.format_exc().rstrip().splitlines():
            print("   ", line)

smoke_tokens = None
smoke_tool_calls = None
smoke_phases = {}
if SMOKE_RESULT is not None:
    def _grab(name):
        if isinstance(SMOKE_RESULT, dict):
            return SMOKE_RESULT.get(name)
        return getattr(SMOKE_RESULT, name, None)

    for name in ("total_completion_tokens", "completion_tokens", "prompt_tokens"):
        value = _grab(name)
        if isinstance(value, (int, float)):
            smoke_tokens = int(value)
            break
    for name in ("tool_calls_used", "tool_calls", "total_tool_calls"):
        value = _grab(name)
        if isinstance(value, (int, float)):
            smoke_tool_calls = int(value)
            break
    smoke_phases = extract_phase_timings(SMOKE_RESULT)

SMOKE_TOKENS = smoke_tokens if smoke_tokens is not None else 0
SMOKE_TOOL_CALLS = smoke_tool_calls if smoke_tool_calls is not None else 0

SMOKE_REPORT = {
    "task_id": task_field(tasks[0], "instance_id") or task_field(tasks[0], "task_id"),
    "repo": task_field(tasks[0], "repo"),
    "total_completion_tokens": smoke_tokens,
    "tool_calls_used": smoke_tool_calls,
    "phase_timings": smoke_phases,
    "result_type": type(SMOKE_RESULT).__name__ if SMOKE_RESULT is not None else None,
    "evaluator_error": evaluator_error,
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
}
(WORKING_DIR / "nondegeneracy_smoke.json").write_text(json.dumps(SMOKE_REPORT, indent=2, sort_keys=True) + "\n")

print("=" * 78)
print("  NON-DEGENERACY SMOKE")
print("=" * 78)
print("  task                    :", SMOKE_REPORT["task_id"], "| repo", SMOKE_REPORT["repo"])
print("  result type             :", SMOKE_REPORT["result_type"])
print("  total_completion_tokens :", smoke_tokens, " (must be > 0)")
print("  tool_calls_used         :", smoke_tool_calls, " (must be >= 1)")
print("  phase timings (real)    :", json.dumps(smoke_phases, sort_keys=True))
print("  " + "=" * 74)

# The abort marker. Its ABSENCE is what Phase 5 and the reader look for.
DEGENERATE = (
    (SMOKE_RESULT is None)
    or (SMOKE_TOKENS <= 0)
    or (SMOKE_TOOL_CALLS < 1)
)

if DEGENERATE and not ALLOW_PARTIAL_PREFLIGHT:
    print()
    print("#" * 78)
    print("#   D E G E N E R A T E   --   S M O K E   T O K E N S   0".center(78))
    print("#" * 78)
    print("#  The one-task smoke did not produce a usable generation:")
    if SMOKE_RESULT is None:
        print("#    - no result object came back at all")
        print("#    - evaluator_error:", evaluator_error)
    if SMOKE_TOKENS <= 0:
        print(f"#    - total_completion_tokens = {SMOKE_TOKENS} (expected > 0)")
    if SMOKE_TOOL_CALLS < 1:
        print(f"#    - tool_calls_used = {SMOKE_TOOL_CALLS} (expected >= 1)")
    print("#")
    print("#  This is the EXACT signature of the 0/129 run: the model never")
    print("#  generates. Running 20 tasks now would cost ~1.7 h of free quota to")
    print("#  confirm what one task already proved.")
    print("#")
    print("#  Check in this order:")
    print("#    1. Phase 4.5 sampling_yaml_has_no_thinking_level -- a thinking_level")
    print("#       key makes LiteLLM send reasoning_effort and the Gemma bridge")
    print("#       raises RuntimeError on turn 1.")
    print("#    2. Model access: does the Kaggle notebook output show the vLLM")
    print("#       server bound to the served Gemma 4 alias?")
    print("#    3. Generation config vs thinking_config in configs/sampling.yaml.")
    print("#")
    print("#  Evidence written to:", WORKING_DIR / "nondegeneracy_smoke.json")
    print("#" * 78, flush=True)
    raise SystemExit(1)

print("  smoke is NON-DEGENERATE -- proceeding to the full sample.", flush=True)
print("=" * 78, flush=True)

## Phase 5 — The run loop

Every selected task, no early break. Per task we record the exit code, the patch size, the
completion tokens, the tool calls used, the wall-clock duration, and **any phase timings the
harness actually reports**.

Phase splits are recorded as `null` when the harness does not expose them. They are not
estimated from wall-clock arithmetic: an inferred split looks identical to a measured one in a
results table, and that is exactly how a guess becomes a fact. The smoke run in Phase 4.6
prints which fields are genuinely available on this harness build.

In [ ]:
# ===========================================================================
# PHASE 5 -- Run every selected task. No cap. No early break.
# ===========================================================================
RESULTS_DIR = WORKING_DIR / "results"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CSV_PATH = WORKING_DIR / "per_task_results.csv"
JSONL_PATH = WORKING_DIR / "per_task_results.jsonl"

CSV_FIELDS = [
    "task_id", "repo", "resolved", "test_exit_code",
    "total_completion_tokens", "tool_calls_used", "patch_size",
    "setup_s", "agent_s", "verify_s", "duration_seconds",
    "error_summary",
]


def run_sync(fn, *args, **kwargs):
    """Call fn, awaiting it if it returns a coroutine. Works inside or outside a loop."""
    import asyncio
    import concurrent.futures

    def _invoke():
        return fn(*args, **kwargs)

    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(_invoke()) if asyncio.iscoroutinefunction(fn) else _invoke()

    # Already inside a running loop: hand off to a worker thread with its own loop.
    with concurrent.futures.ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(lambda: asyncio.run(_invoke())).result()


# Reuse the evaluator the smoke run already proved works.
assert evaluator is not None, "Phase 4.6 must pass before Phase 5 runs"


def _read(result, *names):
    """First non-None value among `names`, across dict and object results."""
    for name in names:
        value = result.get(name) if isinstance(result, dict) else getattr(result, name, None)
        if value is not None:
            return value
    return None


def _as_int(value, default=-1):
    try:
        return int(value)
    except (TypeError, ValueError):
        return default


def run_one_task(task, index, total):
    """Evaluate one task. Returns a fully populated result dict.

    Never raises: any failure is converted into a recorded row so the campaign
    continues and the failure is preserved as data.
    """
    task_id = task_field(task, "instance_id") or task_field(task, "task_id") or f"task_{index}"
    repo = task_field(task, "repo")
    started = time.time()

    try:
        result = run_sync(evaluator.evaluate_task, task=task, task_index=index, total_tasks=total)
    except Exception as exc:
        # Infrastructure or harness failure -> test_exit_code -1 == "no test result".
        return {
            "task_id": task_id,
            "repo": repo,
            "resolved": False,
            "test_exit_code": -1,
            "total_completion_tokens": 0,
            "tool_calls_used": -1,
            "patch_size": 0,
            "setup_s": None,
            "agent_s": None,
            "verify_s": None,
            "duration_seconds": round(time.time() - started, 2),
            "error_summary": f"{type(exc).__name__}: {exc}"[:500],
        }

    patch = _read(result, "agent_patch", "patch") or ""
    phases = extract_phase_timings(result)
    tokens = _read(result, "total_completion_tokens", "completion_tokens", "prompt_tokens")
    tool_calls = _read(result, "tool_calls_used", "tool_calls", "total_tool_calls")
    exit_code = _read(result, "test_exit_code", "exit_code")
    resolved = _read(result, "resolved")

    return {
        "task_id": task_id,
        "repo": repo,
        "resolved": bool(resolved) if resolved is not None else False,
        "test_exit_code": _as_int(exit_code, -1),
        "total_completion_tokens": _as_int(tokens, 0),
        "tool_calls_used": _as_int(tool_calls, -1),
        "patch_size": len(patch) if isinstance(patch, str) else 0,
        "setup_s": phases.get("setup_s"),
        "agent_s": phases.get("agent_s"),
        "verify_s": phases.get("verify_s"),
        "duration_seconds": round(
            float(_read(result, "duration_seconds") or (time.time() - started)), 2
        ),
        "error_summary": str(_read(result, "error_summary", "error") or "")[:500],
    }


results = []
TOTAL_TASKS = len(tasks)
print("=" * 78)
print(f"  RUNNING {TOTAL_TASKS} TASKS -- every selected task, no early break")
print(f"  model={MODEL_ID}  adapter=NONE")
print(f"  budget: {SHIP_MAX_TOOL_CALLS} calls / {SHIP_MAX_TIME_MINUTES} min / {SHIP_TIMEOUT_SECONDS}s")
print("=" * 78)

# NOTE: `results` is appended for every task in `tasks`. There is intentionally
# no `break`, no `[:N]`, and no failure-triggered exit.
for index, task in enumerate(tasks, start=1):
    row = run_one_task(task, index, TOTAL_TASKS)
    results.append(row)

    with open(JSONL_PATH, "a", encoding="utf-8") as handle:
        handle.write(json.dumps(row) + "\n")

    print(
        f"[{index}/{TOTAL_TASKS}] {row['task_id']:<20} "
        f"repo={str(row['repo']):<12} exit={row['test_exit_code']!s:<5} "
        f"resolved={row['resolved']!s:<6} tokens={row['total_completion_tokens']!s:<7} "
        f"tools={row['tool_calls_used']!s:<4} {row['duration_seconds']}s",
        flush=True,
    )

tokens_positive = sum(1 for r in results if r["total_completion_tokens"] > 0)
print("=" * 78)
print(f"  ALL {len(results)} TASKS PROCESSED")
print(f"  non-degenerate (tokens > 0): {tokens_positive}/{len(results)}")
print("=" * 78, flush=True)

## Phase 6 — Persist per-task results (CSV + JSONL)

Both formats are written so results can be retrieved with
`kaggle kernels output`:

* `per_task_results.csv` — flat, one row per task, the `exit_code` column
  sorted so `-1` rows are easy to find.
* `per_task_results.jsonl` — the same records, append-safe, already flushed
  incrementally during the run.

The CSV is sorted by exit code (ascending) so `-1` — the infrastructure
signature — floats to the top rather than hiding among genuine test failures.

In [ ]:
# ===========================================================================
# PHASE 6 -- Write per-task results to CSV and JSONL.
# ===========================================================================
import csv

# Ensure JSONL is complete even if a crash truncated the incremental flush.
with open(JSONL_PATH, "w", encoding="utf-8") as handle:
    for row in results:
        handle.write(json.dumps(row) + "\n")

# Sort by test_exit_code ascending: -1 (infrastructure) floats to the top.
sorted_rows = sorted(
    results,
    key=lambda r: (r["test_exit_code"] is None, r["test_exit_code"] if r["test_exit_code"] is not None else 0),
)

with open(CSV_PATH, "w", newline="", encoding="utf-8") as handle:
    writer = csv.DictWriter(handle, fieldnames=CSV_FIELDS)
    writer.writeheader()
    for row in sorted_rows:
        writer.writerow({field: row.get(field, "") for field in CSV_FIELDS})

print(f"wrote {CSV_PATH}  ({len(sorted_rows)} rows)")
print(f"wrote {JSONL_PATH} ({len(results)} rows)")

# --- Phase timing summary -----------------------------------------------------
# Sums only over rows where the field is genuinely present. A column that is
# entirely null is reported as such rather than as 0.
print("\n" + "=" * 78)
print("  PHASE TIMING SUMMARY (measured fields only)")
print("=" * 78)
print(f"  {'task_id':<22}{'setup_s':>10}{'agent_s':>10}{'verify_s':>10}{'total_s':>10}")


def _fmt(value):
    return f"{value:>10.1f}" if isinstance(value, (int, float)) else f"{'n/a':>10}"


phase_sums = {"setup_s": 0.0, "agent_s": 0.0, "verify_s": 0.0}
phase_counts = {"setup_s": 0, "agent_s": 0, "verify_s": 0}

for row in sorted_rows:
    print(f"  {row['task_id'][:21]:<22}" + "".join(
        _fmt(row.get(k)) for k in ("setup_s", "agent_s", "verify_s")
    ) + _fmt(row.get("duration_seconds")))
    for key in phase_sums:
        value = row.get(key)
        if isinstance(value, (int, float)):
            phase_sums[key] += float(value)
            phase_counts[key] += 1

print("-" * 78)
for key in ("setup_s", "agent_s", "verify_s"):
    if phase_counts[key]:
        print(f"  SUM {key:<16} = {phase_sums[key]:>10.1f}s   over {phase_counts[key]}/{len(results)} tasks")
    else:
        print(f"  SUM {key:<16} = {'n/a':>10}   0/{len(results)} tasks -- harness exposed no such field")
print(f"  SUM total_s          = {sum(r['duration_seconds'] for r in results):>10.1f}s")
print("=" * 78, flush=True)

## Phase 7 — Exit-code histogram (the single most important output)

This is the measurement the whole notebook exists to produce.

Read it like this:

* **`0` = resolved.** A patch worked.
* **`1` = genuine model failure.** The agent produced a patch, it applied, tests
  ran, tests failed. This is the LoRA/prompt's fault — a real, actionable signal.
* **`-1` = infrastructure failure.** No test result was ever produced. Timeout,
  sandbox crash, context-window overflow, dead endpoint. **Not the model's fault.**

A high `-1` count means the leaderboard score is measuring our plumbing, not our
model. A high `1` count with low `-1` means the plumbing is fine and the LoRA
genuinely is not very good. **Those two worlds need completely different fixes.**

We also print the explicit split so it cannot be misread.

In [ ]:
# ===========================================================================
# PHASE 7 -- Exit-code histogram + non-degeneracy counts.
# The single most important output.
# ===========================================================================
histogram = {}
for row in results:
    key = str(row["test_exit_code"])
    histogram[key] = histogram.get(key, 0) + 1

total = len(results)
resolved_count = sum(1 for r in results if r["resolved"])
exit_minus_one = histogram.get("-1", 0)
exit_zero = histogram.get("0", 0)
exit_one = histogram.get("1", 0)
genuine_failures = exit_one
infrastructure = exit_minus_one

# The degeneracy counters are what this run exists to produce. `tokens_positive`
# below 100% means the model failed to generate on those tasks -- that is a
# SERVING failure, categorically different from exit 1, which is a model failure
# after a successful generation.
tokens_positive = sum(1 for r in results if r["total_completion_tokens"] > 0)
tools_used_tasks = sum(1 for r in results if r["tool_calls_used"] >= 1)
patches_nonempty = sum(1 for r in results if r["patch_size"] > 0)

histogram_payload = {
    "notebook": NOTEBOOK_NAME,
    "kernel_slug": KERNEL_SLUG,
    "generated_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "adapter": None,
    "corpus_size": CORPUS_SIZE,
    "total_tasks": total,
    "resolved_count": resolved_count,
    "resolution_rate_percent": round(resolved_count / total * 100, 2) if total else 0.0,
    "histogram": histogram,
    "interpretation": {
        "0": "resolved: patch applied, tests passed",
        "1": "genuine model failure: patch applied, tests ran and failed",
        "-1": "infrastructure failure: no test result was ever produced",
    },
    "infrastructure_failure_count": infrastructure,
    "genuine_model_failure_count": genuine_failures,
    "infrastructure_failure_percent": round(infrastructure / total * 100, 2) if total else 0.0,
    "degeneracy": {
        "tasks_with_tokens": tokens_positive,
        "tasks_with_zero_tokens": total - tokens_positive,
        "zero_token_percent": round((total - tokens_positive) / total * 100, 2) if total else 0.0,
        "tasks_with_tool_calls": tools_used_tasks,
        "tasks_with_nonempty_patch": patches_nonempty,
        "smoke_tokens": SMOKE_TOKENS,
        "smoke_tool_calls": SMOKE_TOOL_CALLS,
    },
    "phase_timing_totals": {
        key: (round(phase_sums[key], 2) if phase_counts[key] else None)
        for key in ("setup_s", "agent_s", "verify_s")
    },
    "phase_timing_sample_counts": phase_counts,
}

HISTOGRAM_PATH = WORKING_DIR / "exit_code_histogram.json"
HISTOGRAM_PATH.write_text(json.dumps(histogram_payload, indent=2, sort_keys=True) + "\n")

print("=" * 78)
print("  EXIT-CODE HISTOGRAM")
print("=" * 78)
print(f"  total tasks : {total}   (corpus {CORPUS_SIZE})")
print(f"  resolved    : {resolved_count}")
for key in sorted(histogram, key=lambda k: int(k) if k.lstrip('-').isdigit() else 999):
    bar = "#" * max(1, histogram[key])
    print(f"    exit {key:>4} : {histogram[key]:>4}  {bar}")
print("-" * 78)
print(f"  resolved (exit 0)                  : {exit_zero}")
print(f"  genuine model failure (exit 1)     : {genuine_failures}")
print(f"  infrastructure failure (exit -1)   : {infrastructure}  <-- NOT the model")
print("-" * 78)
print(f"  tasks with tokens > 0              : {tokens_positive}/{total}")
print(f"  tasks with ZERO tokens             : {total - tokens_positive}/{total}   <-- SERVING failure")
print(f"  tasks with >=1 tool call           : {tools_used_tasks}/{total}")
print(f"  tasks with a non-empty patch       : {patches_nonempty}/{total}")
print("=" * 78)
print(json.dumps(histogram_payload, indent=2, sort_keys=True))
print(f"\nwrote {HISTOGRAM_PATH}", flush=True)

## Phase 8 — Final summary and machine-readable manifest

The summary answers the original question in one block: *how much of this run was
our infrastructure, and how much was the model?*

`run_manifest.json` captures the run's identity — notebook name, timestamp,
adapter sha256 **as actually observed**, model id, totals, and kernel slug — so a
result can be tied back to the exact inputs that produced it. If the adapter hash
was not what we expected, the manifest still records what was really mounted.

In [ ]:
# ===========================================================================
# PHASE 8 -- Final summary + run manifest.
# ===========================================================================
resolution_rate = (resolved_count / total * 100) if total else 0.0
zero_token_tasks = total - tokens_positive

print("=" * 78)
print("  FINAL SUMMARY -- gemma4-baseline-v1 (adapter-free diagnostic)")
print("=" * 78)
print(f"  tasks attempted   : {total} of {CORPUS_SIZE} corpus")
print(f"  tasks resolved    : {resolved_count}")
print(f"  resolution rate   : {resolution_rate:.2f}%")
print(f"  model             : {MODEL_ID}")
print(f"  adapter           : NONE")
print(f"  submission sha256 : {OBSERVED_SUBMISSION_SHA256}")
print(f"    expected        : {SUBMISSION_ZIP_SHA256}")
if OBSERVED_SUBMISSION_SHA256 != SUBMISSION_ZIP_SHA256:
    print("    *** ARTIFACT MISMATCH: this run did NOT measure the shipped zip ***")
print(f"  zero-token tasks  : {zero_token_tasks}/{total}")
print(f"  tools >=1         : {tools_used_tasks}/{total}")
print(f"  non-empty patches : {patches_nonempty}/{total}")
print("-" * 78)
print("  PREFLIGHT (infrastructure preconditions):")
for check in PREFLIGHT_CHECKS:
    print(f"    [{check['verdict']}] {check['name']}")
print(
    f"    {len(PREFLIGHT_CHECKS) - len(PREFLIGHT_FAILED)}/{len(PREFLIGHT_CHECKS)} passed"
    f", policy={'CONTINUE (operator override)' if ALLOW_PARTIAL_PREFLIGHT else 'ABORT'}"
)
print(f"  evidence          : {PREFLIGHT_REPORT_PATH}")
print(f"  exit-code dist    : {histogram}")
print("-" * 78)

# The verdict branches on the SERVING signal first. A run where the model never
# generated has no quality verdict to give at all, and reporting one anyway is
# how a 0/129 infrastructure failure gets written up as model quality.
if total and zero_token_tasks == total:
    print("  VERDICT: EVERY task returned zero completion tokens.")
    print("  The model never generated. This is a SERVING/configuration failure,")
    print("  NOT model quality, and NOT evidence about the prompt. The smoke")
    print("  run in Phase 4.6 should have caught this before any task ran.")
elif total and zero_token_tasks / total > 0.5:
    print(f"  VERDICT: {zero_token_tasks}/{total} tasks produced no tokens.")
    print("  Generation is broken on the majority of tasks. Fix the serving path")
    print("  (thinking_level, model alias, thinking_config) before drawing any")
    print("  conclusion about the prompt.")
elif exit_one:
    print("  VERDICT: generation is healthy; failures are genuine (exit 1).")
    print("  The prompt and model are the limiting factor -- this is the first")
    print("  regime in which prompt work has anything to act on.")
else:
    print("  VERDICT: insufficient signal. Inspect the histogram and per-task rows.")
print("=" * 78)

# END-OF-RUN banner. By the time anyone reads a score, the override decision
# has been forgotten; this makes the degraded run impossible to misread.
if PREFLIGHT_FAILED and ALLOW_PARTIAL_PREFLIGHT:
    print()
    print("!" * 78)
    print("  FINAL WARNING: THIS RUN HAD FAILED PRECONDITIONS".center(78))
    print("!" * 78)
    print("  ALLOW_PARTIAL_PREFLIGHT was True, so the run continued past broken")
    print("  infrastructure preconditions. THE SCORE ABOVE MAY BE MEANINGLESS")
    print("  and must not be read as model quality. Failed checks:")
    for check in PREFLIGHT_FAILED:
        print(f"    - {check['name']}: observed {check['observed']}")
    print(f"  Full evidence: {PREFLIGHT_REPORT_PATH}")
    print("!" * 78)

MANIFEST_PATH = WORKING_DIR / "run_manifest.json"
manifest = {
    "notebook": NOTEBOOK_NAME,
    "kernel_slug": KERNEL_SLUG,
    "timestamp_utc": RUN_TS,
    "completed_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "model_id": MODEL_ID,
    "adapter": None,
    "submission_zip_sha256_expected": SUBMISSION_ZIP_SHA256,
    "submission_zip_sha256_observed": OBSERVED_SUBMISSION_SHA256,
    "submission_hash_matched": OBSERVED_SUBMISSION_SHA256 == SUBMISSION_ZIP_SHA256,
    "shipped_files": SHIPPED_FILES,
    "shipped_eval_config_evaluation": EVAL_SECTION,
    "corpus_size": CORPUS_SIZE,
    "total_tasks": total,
    "task_limit": TASK_LIMIT,
    "stratified_mix": dict(STRATIFIED_MIX),
    "resolved_count": resolved_count,
    "resolution_rate_percent": round(resolution_rate, 2),
    "exit_code_histogram": histogram,
    "infrastructure_failure_count": infrastructure,
    "degeneracy": histogram_payload["degeneracy"],
    "phase_timing_totals": histogram_payload["phase_timing_totals"],
    "phase_timing_sample_counts": phase_counts,
    "nondegeneracy_smoke": SMOKE_REPORT,
    "tasks_path": str(TASKS_PATH),
    "task_load_method": load_method,
    "wheelhouse_dir": str(PRIMARY_WHEELHOUSE) if PRIMARY_WHEELHOUSE else None,
    "api_surface_file": str(API_SURFACE_PATH),
    "preflight": {
        "all_passed": not PREFLIGHT_FAILED,
        "allow_partial_preflight": ALLOW_PARTIAL_PREFLIGHT,
        "failed_count": len(PREFLIGHT_FAILED),
        "failed_checks": [c["name"] for c in PREFLIGHT_FAILED],
        "checks_passed": len(PREFLIGHT_CHECKS) - len(PREFLIGHT_FAILED),
        "checks_total": len(PREFLIGHT_CHECKS),
        "checks": PREFLIGHT_CHECKS,
        "report_file": str(PREFLIGHT_REPORT_PATH),
    },
}
MANIFEST_PATH.write_text(json.dumps(manifest, indent=2, sort_keys=True) + "\n")

print("\nARTIFACTS WRITTEN:")
for path in sorted(WORKING_DIR.glob("*")):
    if path.is_file():
        print(f"  {path.name:<28} {path.stat().st_size:>12,} bytes")
print(f"\nmanifest: {MANIFEST_PATH}", flush=True)